# 05 · Train the queue  (steps 10-12)
**Accelerator: GPU T4 x2.** Run with **Save Version → Save & Run All (Commit)**; you can close the laptop. Re-running continues where the last run stopped, on any account.


Set `QUEUE` and run. Each session trains for ~11 h, saving a checkpoint every 30 min and at every epoch, and
uploads to HF in the background. A half-finished job continues exactly where it stopped. Two T4s train two
different jobs. When this account's GPU quota is used up, run the same notebook on another account (same
`HF_REPO`, its own `HF_TOKEN` secret): it picks up the queue automatically.

Queues: `queues/main.yaml` (main rows, seeds 0→1→2), `queues/ablations.yaml`, `queues/nyud.yaml`,
`queues/multicue.yaml`.

In [ ]:
# ==== SETTINGS (nothing here needs to change between Kaggle accounts) ====
HF_REPO = "sujalprasad90/pedger-mtkd-store"   # the ONE private Hugging Face repo shared by every account
ACCOUNT = ""                                  # optional label, e.g. your name; sessions are told apart automatically
REPO_URL = ""                                 # optional GitHub URL of the code ("" = Kaggle dataset or HF copy)
# Once per Kaggle account, and for every imported notebook:
#   Add-ons -> Secrets -> HF_TOKEN = a WRITE token from the Hugging Face account that owns HF_REPO, ticked for this notebook
#   Settings -> Internet: On;  Settings -> Accelerator: as written in the title
# The preflight cell below checks all of this and stops with a plain-language fix if something is missing.

In [ ]:
QUEUE = "queues/main.yaml"
HOURS = 11.25            # whole-session budget, counted from the first cell (Kaggle limit: 12 h)

In [ ]:
# ---- setup: secrets, code, dependencies (no edits needed) ----
import glob, os, shutil, subprocess, sys, time
SESSION_START = time.time()
def sh(cmd, cwd=None):
    """Run a shell command and show its output; stop the notebook if it fails (a plain `!cmd` carries on)."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for raw in p.stdout:                       # progress bars (\r updates) are shown as their last state
        print(raw.decode(errors="replace").rstrip("\r\n").split("\r")[-1], flush=True)
    if p.wait():
        raise RuntimeError(f"this step failed (exit code {p.returncode}); its error is printed above:\n  {cmd}")
os.environ["PEDGER_SESSION_START"] = str(SESSION_START)   # long steps stop and save before Kaggle's 12-hour limit
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Secret HF_TOKEN is not available: Add-ons -> Secrets -> add HF_TOKEN and tick it for THIS "
                       "notebook, then run again.") from e
os.environ["PEDGER_REMOTE"] = "hf:" + HF_REPO
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.25,<2.0", "pyyaml"], check=True)
W, REPO, WS = "/kaggle/working", "/kaggle/working/pedger-mtkd", "/kaggle/working/ws"
# Always start from the newest code: an interactive session can still hold an old copy from an earlier run.
os.chdir(W)
shutil.rmtree(REPO, ignore_errors=True)
shutil.rmtree(f"{W}/hf_code", ignore_errors=True)
for _m in [m for m in list(sys.modules) if m == "pedger_kd" or m.startswith("pedger_kd.")]:
    del sys.modules[_m]
attached = sorted(glob.glob("/kaggle/input/*/pedger_kd/__init__.py") + glob.glob("/kaggle/input/*/*/pedger_kd/__init__.py"))
if attached:                                       # code uploaded as a Kaggle Dataset (pedger-mtkd-code)
    shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), REPO)
    CODE_SOURCE = "kaggle dataset"
elif REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
    CODE_SOURCE = "git"
else:                                              # the copy kept on Hugging Face
    from huggingface_hub import snapshot_download
    snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["code/*"], local_dir=f"{W}/hf_code",
                      token=os.environ["HF_TOKEN"])
    shutil.copytree(f"{W}/hf_code/code", REPO)
    CODE_SOURCE = "huggingface"
_stamp = os.path.join(REPO, "CODE_STAMP.json")
print("code from:", CODE_SOURCE, "| version:", open(_stamp).read().split('"')[3] if os.path.exists(_stamp) else "unstamped")
os.chdir(REPO)
sys.path.insert(0, REPO)
sh("python tools/fetch_upstream.py")
# preflight: session type, GPUs, internet, Hugging Face account and repo access
if subprocess.run([sys.executable, "-m", "pedger_kd.preflight"]).returncode:
    raise RuntimeError("Preflight failed: fix the PROBLEM lines printed above, then run the notebook again.")

In [ ]:
!python -m pedger_kd.sync status

In [ ]:
!python -m pedger_kd.jobs --queue {QUEUE} --ws {WS} --hours {HOURS} --session-start {SESSION_START} --gpus auto --label '{ACCOUNT}' --squash

In [ ]:
!python -m pedger_kd.sync status

In [ ]:
# keep the notebook output small (everything important is already on Hugging Face)
for d in ["ws/data", "ws/teachers", "ws/.staging", "hf_code", "work", "BSR"]:
    shutil.rmtree(f"/kaggle/working/{d}", ignore_errors=True)
print(f"session used {(time.time() - SESSION_START) / 3600:.2f} h")